# Clinical Recruitment GRPO Training (Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratimassaravanan/clinical-recruitment-env/blob/main/notebooks/colab_grpo_training.ipynb)

Trains a 4-bit LoRA model with **Unsloth + TRL GRPO** against the live
Adaptive Clinical Recruitment environment on HF Spaces.

**Requires T4 GPU runtime.** Go to *Runtime → Change runtime type → T4 GPU*
before running.

In [ ]:
# Cell 2 — Install dependencies
!pip install -q --upgrade pip
!pip install -q unsloth
!pip install -q --no-deps "trl>=0.19.0"
!pip install -q git+https://github.com/huggingface/transformers.git@main
!pip install -q "datasets>=2.21.0" "accelerate>=0.34.0" "openenv-core>=0.2.1"

In [ ]:
# Cell 3 — Imports + GPU verification
import json, os, pathlib, torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from openenv import GenericEnvClient

assert torch.cuda.is_available(), "No CUDA GPU found — enable T4 in Runtime settings!"
gpu = torch.cuda.get_device_name(0)
assert "T4" in gpu, f"Expected T4 GPU but got: {gpu}"
print(f"GPU: {gpu} | CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

In [ ]:
# Cell 4 — Configuration
ENV_URL = os.getenv("ENV_URL", "https://pratimassaravanan-clinical-recruitment.hf.space")
MODEL_NAME = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"
TASKS = ["easy_bench", "medium_bench", "hard_bench"]

MAX_SEQ = 2048
LORA_R = 16
LORA_ALPHA = 16

NUM_GEN = 4
MAX_COMP = 1024
BATCH = 2
GRAD_ACC = 4
EPOCHS = 1
LR = 5e-6

OUTPUT_DIR = "grpo_clinical_recruitment"
RESULTS_DIR = pathlib.Path("data/training_outputs")

SYSTEM_PROMPT = (
    "You are a long-horizon clinical trial recruitment agent. "
    "Use tools to manage screening, recontact, allocation, planning, and memory."
)

print(f"Env: {ENV_URL}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# Cell 5 — Load model with Unsloth FastLanguageModel 4-bit + LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

In [ ]:
# Cell 6 — ClinicalRecruitmentToolEnv
_H: str = "noise_dominant"
_C: float = 0.6


class ClinicalRecruitmentToolEnv:
    """OpenEnv tool-call environment for Clinical Recruitment.

    Wraps the remote HF Spaces environment via GenericEnvClient and
    exposes 8 tool methods that the GRPO trainer can call.
    """

    def __init__(self) -> None:
        self.client = GenericEnvClient(base_url=ENV_URL).sync()
        self.reward: float = 0.0
        self.initial_budget: float = 0.0
        self.done: bool = False
        self.last_result = None
        self.last_observation = None
        self.action_history: list[str] = []
        self.enrollment_history: list[int] = []
        self.budget_history: list[float] = []
        self.hypothesis_history: list[str] = []

    def reset(self, **kw) -> str:
        """Reset the environment. Called by the trainer lifecycle."""
        self.client.connect()
        self.last_result = self.client.reset(
            task=kw.get("task") or kw.get("task_id") or "easy_bench"
        )
        self.last_observation = self.last_result.observation
        self.reward = 0.0
        self.done = bool(self.last_result.done)
        self.action_history = []
        self.enrollment_history = []
        self.budget_history = []
        self.hypothesis_history = []
        obs = self.last_observation or {}
        self.initial_budget = obs.get("budget_remaining", 0.0)
        self.enrollment_history.append(obs.get("enrolled_so_far", 0))
        self.budget_history.append(obs.get("budget_remaining", 0.0))
        return self._fmt()

    def close(self) -> None:
        """Close the client connection. Called by the trainer lifecycle."""
        try:
            self.client.close()
        except Exception:
            pass

    # ── Tool methods (8 actions) ─────────────────────────────────────

    def screen_patient(
        self,
        patient_id: str,
        hypothesis: str = _H,
        confidence: float = _C,
    ) -> str:
        """Screen a candidate patient for eligibility.

        Args:
            patient_id: Patient id from available_patients.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step(dict(
            action_type="screen_patient",
            patient_id=patient_id,
            hypothesis=hypothesis,
            confidence=confidence,
        ))

    def recontact(
        self,
        patient_id: str,
        hypothesis: str = _H,
        confidence: float = _C,
    ) -> str:
        """Recontact an eligible patient for consent or enrollment.

        Args:
            patient_id: Patient id from recontact_candidates.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step(dict(
            action_type="recontact",
            patient_id=patient_id,
            hypothesis=hypothesis,
            confidence=confidence,
        ))

    def allocate_to_site(
        self,
        patient_id: str,
        site_id: str,
        hypothesis: str = _H,
        confidence: float = _C,
    ) -> str:
        """Allocate a consented patient to a recruitment site.

        Args:
            patient_id: Patient id from allocation_candidates.
            site_id: Site id from site_performance.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step(dict(
            action_type="allocate_to_site",
            patient_id=patient_id,
            site_id=site_id,
            hypothesis=hypothesis,
            confidence=confidence,
        ))

    def adjust_strategy(
        self,
        strategy_change: str,
        hypothesis: str = _H,
        confidence: float = _C,
    ) -> str:
        """Adjust recruitment strategy such as increase_outreach or negotiate_site_A.

        Args:
            strategy_change: Strategy name like increase_outreach or tighten_criteria.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step(dict(
            action_type="adjust_strategy",
            strategy_change=strategy_change,
            hypothesis=hypothesis,
            confidence=confidence,
        ))

    def plan_next_phase(
        self,
        target_phase: str,
        plan_summary: str = "advance the bottleneck",
    ) -> str:
        """Create or revise the current high-level recruitment plan.

        Args:
            target_phase: One of screening, conversion, allocation, retention, recovery.
            plan_summary: Natural-language summary of the plan.
        """
        return self._step(dict(
            action_type="plan_next_phase",
            target_phase=target_phase,
            plan_summary=plan_summary,
        ))

    def summarize_and_index(
        self,
        memory_key: str,
        memory_payload: str,
    ) -> str:
        """Write a summary into indexed episodic memory.

        Args:
            memory_key: Key for the indexed memory item.
            memory_payload: Summary text to store.
        """
        return self._step(dict(
            action_type="summarize_and_index",
            memory_key=memory_key,
            memory_payload=memory_payload,
        ))

    def retrieve_relevant_history(
        self,
        memory_query: str,
    ) -> str:
        """Retrieve indexed memory entries relevant to the current bottleneck.

        Args:
            memory_query: Query string for indexed memory retrieval.
        """
        return self._step(dict(
            action_type="retrieve_relevant_history",
            memory_query=memory_query,
        ))

    def stop_recruitment(self) -> str:
        """End the current recruitment episode early."""
        return self._step(dict(action_type="stop_recruitment"))

    # ── Internal helpers ─────────────────────────────────────────────

    def _step(self, action: dict) -> str:
        if self.done:
            raise ValueError("Episode finished.")
        self.last_result = self.client.step(action)
        self.last_observation = self.last_result.observation
        self.reward = float(self.last_result.reward or 0)
        self.done = bool(self.last_result.done)
        self.action_history.append(action.get("action_type", ""))
        obs = self.last_observation or {}
        self.enrollment_history.append(obs.get("enrolled_so_far", 0))
        self.budget_history.append(obs.get("budget_remaining", 0))
        if h := action.get("hypothesis"):
            self.hypothesis_history.append(h)
        return self._fmt()

    def _fmt(self) -> str:
        o = self.last_observation or {}
        return (
            f"step={o.get('timestamp')} budget={o.get('budget_remaining')} "
            f"enrolled={o.get('enrolled_so_far')}/{o.get('target_enrollment')} "
            f"avail={len(o.get('available_patients', []))} "
            f"recontact={len(o.get('recontact_candidates', []))} "
            f"allocate={len(o.get('allocation_candidates', []))} "
            f"funnel={o.get('current_funnel', {})}"
        )

In [ ]:
# Cell 7 — Reward functions (5 components)

def reward_enrollment_progress(environments, **_):
    """Fraction of target enrollment reached."""
    return [
        min(
            1.0,
            (e.last_observation or {}).get("enrolled_so_far", 0)
            / max(1, (e.last_observation or {}).get("target_enrollment", 100)),
        )
        for e in environments
    ]


def reward_budget_efficiency(environments, **_):
    """Enrollment per unit budget spent."""
    results = []
    for e in environments:
        o = e.last_observation or {}
        ib = e.initial_budget or 1
        spent = max(0, ib - o.get("budget_remaining", 0))
        t = max(1, o.get("target_enrollment", 100))
        if spent > 0:
            results.append(min(1.0, (o.get("enrolled_so_far", 0) / t) / (spent / ib)))
        else:
            results.append(0.0)
    return results


def reward_screening_accuracy(environments, **_):
    """Enrolled-to-screened ratio minus dropout penalty."""
    results = []
    for e in environments:
        f = (e.last_observation or {}).get("current_funnel", {})
        s = f.get("screened", 0)
        if s > 0:
            results.append(
                max(0, min(1, f.get("enrolled", 0) / s - 0.5 * f.get("dropped", 0) / s))
            )
        else:
            results.append(0.0)
    return results


def reward_action_diversity(environments, **_):
    """Fraction of 8 action types used."""
    return [
        min(1, len(set(e.action_history)) / 8) if e.action_history else 0.0
        for e in environments
    ]


def reward_hypothesis_consistency(environments, **_):
    """Penalizes switching, rewards correct world-type match."""
    results = []
    for e in environments:
        hs = e.hypothesis_history
        if len(hs) < 2:
            results.append(0.5)
            continue
        sw = sum(1 for i in range(1, len(hs)) if hs[i] != hs[i - 1])
        con = 1.0 if sw <= 1 else max(0.2, 1 - (sw - 1) * 0.2)
        wt = (e.last_observation or {}).get("world_type", "")
        bonus = (
            0.2
            if {
                "dropout_dominant": "dropout",
                "noise_dominant": "noise",
                "site_bias": "site_bias",
            }.get(hs[-1], "")
            == wt
            and wt
            else 0
        )
        results.append(min(1, con * 0.8 + bonus))
    return results


REWARD_FUNCS = [
    reward_enrollment_progress,
    reward_budget_efficiency,
    reward_screening_accuracy,
    reward_action_diversity,
    reward_hypothesis_consistency,
]

In [ ]:
# Cell 8 — Evaluation helpers: run_episode + _parse_action

def _parse_action(resp: str, obs: dict) -> dict:
    """Map a model response string to an environment action dict."""
    if "screen" in resp and obs.get("available_patients"):
        return dict(
            action_type="screen_patient",
            patient_id=obs["available_patients"][0]["id"],
            hypothesis=_H,
            confidence=_C,
        )
    if "allocate" in resp and obs.get("allocation_candidates") and obs.get("site_performance"):
        return dict(
            action_type="allocate_to_site",
            patient_id=obs["allocation_candidates"][0]["id"],
            site_id=list(obs["site_performance"])[0],
            hypothesis=_H,
            confidence=_C,
        )
    if "recontact" in resp and obs.get("recontact_candidates"):
        return dict(
            action_type="recontact",
            patient_id=obs["recontact_candidates"][0]["id"],
            hypothesis=_H,
            confidence=_C,
        )
    if "adjust" in resp:
        return dict(
            action_type="adjust_strategy",
            strategy_change="increase_outreach",
            hypothesis=_H,
            confidence=_C,
        )
    if "plan" in resp:
        return dict(
            action_type="plan_next_phase",
            target_phase="screening",
            plan_summary="screen more",
        )
    if "stop" in resp:
        return dict(action_type="stop_recruitment")
    # Fallback: screen if patients available, else adjust
    if obs.get("available_patients"):
        return dict(
            action_type="screen_patient",
            patient_id=obs["available_patients"][0]["id"],
            hypothesis=_H,
            confidence=_C,
        )
    return dict(
        action_type="adjust_strategy",
        strategy_change="increase_outreach",
        hypothesis=_H,
        confidence=_C,
    )


def run_episode(mdl, tok, task: str = "easy_bench", n: int = 30) -> dict:
    """Run a single evaluation episode and return metrics."""
    env = ClinicalRecruitmentToolEnv()
    try:
        obs_text = env.reset(task=task)
    except Exception as e:
        return {"task": task, "error": str(e)}

    FastLanguageModel.for_inference(mdl)
    total_r, steps = 0.0, 0

    for _ in range(n):
        if env.done:
            break
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"State: {obs_text}\nChoose next action."},
        ]
        inp = tok(
            tok.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
            ),
            return_tensors="pt",
        ).to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**inp, max_new_tokens=256, do_sample=False)
        resp = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).lower()
        try:
            obs_text = env._step(_parse_action(resp, env.last_observation or {}))
            total_r += env.reward
            steps += 1
        except Exception:
            break

    fo = env.last_observation or {}
    res = {
        "task": task,
        "actions": steps,
        "total_reward": round(total_r, 4),
        "enrolled": fo.get("enrolled_so_far", 0),
        "target": fo.get("target_enrollment", 100),
    }
    for fn in REWARD_FUNCS:
        res[fn.__name__.replace("reward_", "")] = round(fn([env])[0], 4)
    env.close()
    return res

In [ ]:
# Cell 9 — BEFORE training evaluation on all 3 tasks
sep = "=" * 60

print(f"\n{sep}")
print("BEFORE TRAINING")
print(sep)

before = {}
for t in TASKS:
    r = run_episode(model, tokenizer, t)
    before[t] = r
    print(f"  [{t}] enrolled={r.get('enrolled', 0)}/{r.get('target', 0)} "
          f"reward={r.get('total_reward', 0)}")

In [ ]:
# Cell 10 — Build dataset + GRPOTrainer + train
ds = Dataset.from_dict({
    "prompt": [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Improve recruitment."},
        ]
        for _ in range(48)
    ],
    "task_id": [TASKS[i % 3] for i in range(48)],
})

FastLanguageModel.for_training(model)

cfg = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_generations=NUM_GEN,
    max_completion_length=MAX_COMP,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    optim="adamw_8bit",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    chat_template_kwargs={"enable_thinking": False},
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds,
    reward_funcs=REWARD_FUNCS,
    environment_factory=ClinicalRecruitmentToolEnv,
    args=cfg,
)

print(f"\n{sep}")
print(f"GRPO TRAINING on {gpu}")
print(sep)

trainer.train()

In [ ]:
# Cell 11 — AFTER training evaluation + comparison table
print(f"\n{sep}")
print("AFTER TRAINING")
print(sep)

after = {}
for t in TASKS:
    r = run_episode(model, tokenizer, t)
    after[t] = r
    print(f"  [{t}] enrolled={r.get('enrolled', 0)}/{r.get('target', 0)} "
          f"reward={r.get('total_reward', 0)}")

CMP = [
    "total_reward",
    "enrolled",
    "enrollment_progress",
    "budget_efficiency",
    "screening_accuracy",
    "action_diversity",
    "hypothesis_consistency",
]

print(f"\n{sep}")
print("COMPARISON")
print(sep)
for t in TASKS:
    b, a = before.get(t, {}), after.get(t, {})
    print(f"\n  [{t}]")
    for k in CMP:
        bv, av = b.get(k, 0), a.get(k, 0)
        if isinstance(bv, (int, float)) and isinstance(av, (int, float)):
            delta = av - bv
            sign = "+" if delta >= 0 else ""
            print(f"    {k:>25}  {bv:>10.4f}  {av:>10.4f}  {sign}{delta:>9.4f}")

In [ ]:
# Cell 12 — Save model (LoRA + merged) + results JSON
lora_path = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

merged = model.merge_and_unload()
merged_path = f"{OUTPUT_DIR}/merged_model"
merged.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print(f"LoRA adapter -> {lora_path}")
print(f"Merged model  -> {merged_path}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results_path = RESULTS_DIR / "grpo_results.json"
results_path.write_text(
    json.dumps(
        {
            "before": before,
            "after": after,
            "model": MODEL_NAME,
            "env_url": ENV_URL,
            "gpu": gpu,
        },
        indent=2,
    )
)
print(f"Results JSON  -> {results_path}")

## Honest Caveats

- This is a **hackathon starter notebook**, not a production training pipeline.
- The live Space serves 3 tasks (`easy_bench`, `medium_bench`, `hard_bench`) with 180 max steps each.
- Training against a shared remote Space is subject to **latency, rate limits, and concurrent-user contention**. For serious runs, duplicate the Space or run the environment locally.
- The **0.6B model** is intentionally small for fast iteration on a free T4. Real benchmark quality requires larger models and longer training.
- The multi-component reward decomposition is a principled starting point, but the **weights will need tuning** for your use case.
- Single-epoch training with 48 prompts is a **minimal proof-of-concept**. Expect modest delta between before/after.
- The `_parse_action` heuristic is deliberately simple; a production agent should emit structured tool calls, not keyword-matched free text.